# Football Betting Market Analysis

This notebook is the main working notebook for the closing-odds pipeline.

It is designed to work with the existing Python modules in the project, so the notebook stays clean while the reusable logic lives in `.py` files.

## Pipeline covered here

1. Load project modules
2. Preprocess raw closing odds
3. Build engineered features
4. Save processed data
5. Create a time-based train-test split
6. Run baseline backtests
7. Inspect evaluation tables
8. Prepare for probability calibration


## Recommended project structure

```text
project_root/
│
├── data/
│   ├── raw/
│   └── processed/
│
├── results/
│   └── closing_backtest/
│
├── src/
│   └── betproj/
│       ├── data_loader.py
│       ├── preprocess_closing.py
│       ├── features_closing.py
│       └── backtest.py
│
├── scripts/
│   ├── build_closing_data.py
│   └── run_closing_backtest.py
│
└── notebooks/
    └── 01_closing_pipeline.ipynb
```

The notebook should orchestrate the workflow, while the real logic remains in the package modules.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.append(str(PROJECT_ROOT / 'src'))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'closing_backtest'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOKS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Import project modules

These imports connect the notebook to the modular backend.

In [ ]:
from betproj.data_loader import load_closing_odds
from betproj.preprocess_closing import preprocess_closing_odds
from betproj.features_closing import build_closing_features
from betproj.backtest import select_bets, threshold_grid_backtest, evaluation_tables


## Step 1 — Load raw closing odds

This is the raw historical closing-odds dataset.

In [ ]:
df_raw = load_closing_odds()
print(df_raw.shape)
df_raw.head()

## Step 2 — Preprocess

This parses the match date and adds the categorical match result:

- `H` = home win
- `D` = draw
- `A` = away win


In [ ]:
df_pre = preprocess_closing_odds()
print(df_pre.shape)
df_pre[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].head()

## Step 3 — Feature engineering

This builds the core closing features:

- normalized implied probabilities
- expected values
- profit columns
- value-gap features


In [ ]:
df = build_closing_features(df_pre)
print(df.shape)
df.head()

## Step 4 — Save processed features

This keeps the pipeline reproducible and avoids recomputing features each time.

In [ ]:
processed_path = DATA_PROCESSED / 'closing_features.parquet'
df.to_parquet(processed_path, index=False)
processed_path

## Step 5 — Time-based train-test split

For a betting project, the split must be chronological, not random.

This avoids look-ahead bias.

Default split used here:

- train: up to 2012
- test: 2013 onward


In [ ]:
df['match_date'] = pd.to_datetime(df['match_date'])
df = df.sort_values('match_date').reset_index(drop=True)
df['year'] = df['match_date'].dt.year

train = df[df['year'] <= 2012].copy()
test = df[df['year'] >= 2013].copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Train years:', train['year'].min(), '-', train['year'].max())
print('Test years:', test['year'].min(), '-', test['year'].max())

In [ ]:
train_path = DATA_PROCESSED / 'train_closing.parquet'
test_path = DATA_PROCESSED / 'test_closing.parquet'

train.to_parquet(train_path, index=False)
test.to_parquet(test_path, index=False)

train_path, test_path

## Step 6 — Training-set baseline backtest

These are only development results.

Use the training set to explore thresholds and filters, then lock them before evaluating on the test set.

In [ ]:
train_bets = select_bets(
    df=train,
    ev_threshold=0.08,
    mode='best_per_match',
    min_n_odds=10,
    max_odds_allowed=8,
)

train_summary = evaluation_tables(train_bets)['summary']
train_summary

## Threshold grid on training set

This is where you tune the EV threshold using only training data.

In [ ]:
thresholds = [0.00, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]

train_grid = threshold_grid_backtest(
    df=train,
    thresholds=thresholds,
    mode='best_per_match',
    min_n_odds=10,
    max_odds_allowed=8,
)

train_grid

## Step 7 — Freeze parameters and evaluate on test set

After selecting a threshold on the training set, apply exactly the same parameters to the test set.

In [ ]:
chosen_threshold = 0.08

test_bets = select_bets(
    df=test,
    ev_threshold=chosen_threshold,
    mode='best_per_match',
    min_n_odds=10,
    max_odds_allowed=8,
)

test_tables = evaluation_tables(test_bets)
test_tables['summary']

In [ ]:
test_tables['by_year'].head(20)

In [ ]:
test_tables['by_league'].head(20)

## Step 8 — Save selected outputs

This keeps the notebook aligned with the script-based workflow.

In [ ]:
train_grid.to_csv(RESULTS_DIR / 'train_threshold_grid.csv', index=False)
test_tables['summary'].to_csv(RESULTS_DIR / 'test_summary.csv', index=False)
test_tables['by_year'].to_csv(RESULTS_DIR / 'test_by_year.csv', index=False)
test_tables['by_league'].to_csv(RESULTS_DIR / 'test_by_league.csv', index=False)
test_tables['by_outcome'].to_csv(RESULTS_DIR / 'test_by_outcome.csv', index=False)
test_tables['by_bookie'].to_csv(RESULTS_DIR / 'test_by_bookie.csv', index=False)
test_tables['by_n_odds_bin'].to_csv(RESULTS_DIR / 'test_by_n_odds_bin.csv', index=False)
test_bets.to_parquet(RESULTS_DIR / 'test_selected_bets.parquet', index=False)

print('Saved outputs to:', RESULTS_DIR)

## Next notebook section to add later

The natural next step is probability calibration:

1. Fit calibration on the training set
2. Apply calibrated probabilities to the test set
3. Recompute EV
4. Repeat the backtest

That should become the next notebook: `02_probability_calibration.ipynb`.